# PDF 기반 RAG(헌법 Q&A 챗봇)

## 환경 설정

In [2]:
# 필요한 모듈 / 함수 / 클래스 임포트
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

C:\Users\playdata2\AppData\Local\Temp\ipykernel_7004\4170936301.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [3]:
# .env file -> API Key 가져오기
env_file_path='.env'
output = load_dotenv(env_file_path)
print(output)

True


## 데이터 로드 및 변환

In [ ]:
# pdf 파일 경로 설정
pdf_file_path = '대한민국헌법(헌법)(제00010호)(19880225).pdf'

# PDF 파일 로드 및 Document 자료형으로 변환해주는 도구 생성
loader_pdf = PyMuPDFLoader(file_path=pdf_file_path)

# 로드 및 변환 실행 -> 전용 함수 -> load() 함수 -> 결과물: 리스트 [Document, Document, ..., Document]
documents_pdf = loader_pdf.load()

# 결과 확인
print(f'Document 페이지 수: {len(documents_pdf)}개')
print('-' * 80)
print(f'1페이지:\n{documents_pdf[0].page_content}')

Document 페이지 수: 14개
--------------------------------------------------------------------------------
1페이지:
법제처                                                            1                                                       국가법령정보센터
대한민국헌법
 
대한민국헌법
[시행 1988. 2. 25.] [헌법 제10호, 1987. 10. 29., 전부개정]
       전문
 
유구한 역사와 전통에 빛나는 우리 대한국민은 3ㆍ1운동으로 건립된 대한민국임시정부의 법통과 불의에 항거한
4ㆍ19민주이념을 계승하고, 조국의 민주개혁과 평화적 통일의 사명에 입각하여 정의ㆍ인도와 동포애로써 민족의 단
결을 공고히 하고, 모든 사회적 폐습과 불의를 타파하며, 자율과 조화를 바탕으로 자유민주적 기본질서를 더욱 확고
히 하여 정치ㆍ경제ㆍ사회ㆍ문화의 모든 영역에 있어서 각인의 기회를 균등히 하고, 능력을 최고도로 발휘하게 하며
, 자유와 권리에 따르는 책임과 의무를 완수하게 하여, 안으로는 국민생활의 균등한 향상을 기하고 밖으로는 항구적
인 세계평화와 인류공영에 이바지함으로써 우리들과 우리들의 자손의 안전과 자유와 행복을 영원히 확보할 것을 다
짐하면서 1948년 7월 12일에 제정되고 8차에 걸쳐 개정된 헌법을 이제 국회의 의결을 거쳐 국민투표에 의하여 개정
한다.
 
1987년 10월 29일
 
       제1장 총강
 
제1조 ①대한민국은 민주공화국이다.
②대한민국의 주권은 국민에게 있고, 모든 권력은 국민으로부터 나온다.
 
제2조 ①대한민국의 국민이 되는 요건은 법률로 정한다.
②국가는 법률이 정하는 바에 의하여 재외국민을 보호할 의무를 진다.
 
제3조 대한민국의 영토는 한반도와 그 부속도서로 한다.
 
제4조 대한민국은 통일을 지향하며, 자유민주적 기본질서에 입각한 평화적 통일정책을 수립하

## 텍스트 분할

In [5]:
# 텍스트 분할 도구 생성
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],
    chunk_size=500,
    chunk_overlap=0
)

# 문서 분할 -> 전용 함수 -> split_documents() 함수 -> 분할의 기준을 만족하는 세분화된 Document 리스트 생성
docs = text_splitter.split_documents(documents=documents_pdf)

# 결과 확인
print(f'세분화된 Document 개수: {len(docs)}개')
print('-'*80)
print(f'세분화된 첫 번째 Document: \n{docs[0].page_content}')
print('-'*80)
print(f'세분화된 두 번째 Document: \n{docs[1].page_content}')

세분화된 Document 개수: 52개
--------------------------------------------------------------------------------
세분화된 첫 번째 Document: 
법제처                                                            1                                                       국가법령정보센터
대한민국헌법
 
대한민국헌법
[시행 1988. 2. 25.] [헌법 제10호, 1987. 10. 29., 전부개정]
       전문
 
유구한 역사와 전통에 빛나는 우리 대한국민은 3ㆍ1운동으로 건립된 대한민국임시정부의 법통과 불의에 항거한
4ㆍ19민주이념을 계승하고, 조국의 민주개혁과 평화적 통일의 사명에 입각하여 정의ㆍ인도와 동포애로써 민족의 단
결을 공고히 하고, 모든 사회적 폐습과 불의를 타파하며, 자율과 조화를 바탕으로 자유민주적 기본질서를 더욱 확고
히 하여 정치ㆍ경제ㆍ사회ㆍ문화의 모든 영역에 있어서 각인의 기회를 균등히 하고, 능력을 최고도로 발휘하게 하며
--------------------------------------------------------------------------------
세분화된 두 번째 Document: 
, 자유와 권리에 따르는 책임과 의무를 완수하게 하여, 안으로는 국민생활의 균등한 향상을 기하고 밖으로는 항구적
인 세계평화와 인류공영에 이바지함으로써 우리들과 우리들의 자손의 안전과 자유와 행복을 영원히 확보할 것을 다
짐하면서 1948년 7월 12일에 제정되고 8차에 걸쳐 개정된 헌법을 이제 국회의 의결을 거쳐 국민투표에 의하여 개정
한다.
 
1987년 10월 29일
 
       제1장 총강
 
제1조 ①대한민국은 민주공화국이다.
②대한민국의 주권은 국민에게 있고, 모든 권력은 국민으로부터 나온다.
 
제2조 ①대한민국의 국민이 되는 요건은 법률로 정한다.
②국가

## 텍스트 임베딩 모델 생성

In [6]:
# OpenAI -> 임베딩 모델 호출

## 임베딩 모델 이름 설정
model_name = "text-embedding-3-small"

## OpenAI 임베딩 모델 생성
model = OpenAIEmbeddings(
    model=model_name
)

## 백터 저장소 및 검색 도구 생성

In [8]:
# 백터 저장소 경로 설정
chroma_path = 'chroma_db_constitution'

# 기존 DB가 존재하면 로드, 없으면 새로 생성
if os.path.exists(chroma_path) and os.listdir(chroma_path):
    vectorstore = Chroma(
        persist_directory=chroma_path,
        # 이전에 사용했던 임베딩 모델을 등록 
        embedding_function=model
    )
    print("기존 벡터 저장소를 성공적으로 불러왔습니다.")
else:
    vectorstore = Chroma.from_documents(
        documents=docs,
        embedding=model,
        persist_directory=chroma_path
    )
    print("신규 벡터 저장소를 생성하고 저장했습니다.")

기존 벡터 저장소를 성공적으로 불러왔습니다.


In [9]:
# 검색 도구 생성 -> 전용 함수 -> as_retriever() 함수 사용
retriever = vectorstore.as_retriever(search_kwargs={'k': 2})

In [10]:
# 검색 도구 성능 테스트
user_query = "국회의원의 임기는 몇 년인가?"
output = retriever.invoke(user_query)
print(f"검색 결과: \n{output}")

검색 결과: 
[Document(id='c1109e8c-7c13-4630-b8ce-12882afb1015', metadata={'trapped': '', 'title': '', 'file_path': '대한민국헌법(헌법)(제00010호)(19880225).pdf', 'keywords': '', 'creator': '', 'subject': '', 'source': '대한민국헌법(헌법)(제00010호)(19880225).pdf', 'creationdate': '2024-04-01T21:26:24+09:00', 'creationDate': "D:20240401212624+09'00'", 'modDate': "D:20240401212624+09'00'", 'format': 'PDF 1.4', 'producer': 'iText 2.1.7 by 1T3XT', 'author': '', 'total_pages': 14, 'page': 4, 'moddate': '2024-04-01T21:26:24+09:00'}, page_content='③국회의원은 그 지위를 남용하여 국가ㆍ공공단체 또는 기업체와의 계약이나 그 처분에 의하여 재산상의 권리ㆍ이\n익 또는 직위를 취득하거나 타인을 위하여 그 취득을 알선할 수 없다.\n \n제47조 ①국회의 정기회는 법률이 정하는 바에 의하여 매년 1회 집회되며, 국회의 임시회는 대통령 또는 국회재적의원\n4분의 1 이상의 요구에 의하여 집회된다.\n②정기회의 회기는 100일을, 임시회의 회기는 30일을 초과할 수 없다.\n③대통령이 임시회의 집회를 요구할 때에는 기간과 집회요구의 이유를 명시하여야 한다.\n \n제48조 국회는 의장 1인과 부의장 2인을 선출한다.\n \n제49조 국회는 헌법 또는 법률에 특별한 규정이 없는 한 재적의원 과반수의 출석과 출석의원 과반수의 찬성으로 의결\n한다. 가부동수인 때에는 부결된 것으로 본다.\n \n제50조 ①국회의 회의는 공개한다. 다만, 출석의원 과반수의 찬성이 있거나 의장이 국가의 안전보장을 위하여

## RAG 체인 생성

### 프롬프트 생성

In [11]:
# 시스템 프롬프트
system_prompt = """
당신은 신뢰할 수 있는 대한민국 헌법 전문 법률 Q&A 도우미입니다.
주어진 헌법 컨텍스트 정보를 바탕으로 사용자의 질문에 정확하고 공손하게 답변해주세요.

규칙:
1. 답변 시 관련된 헌법 조항(예: 제O조 제O항)이 [컨텍스트]에 명시되어 있다면 반드시 조항 번호를 언급해주세요.
2. [컨텍스트]의 내용에 근거해서만 답변해야 하며, 자의적인 해석이나 허위 정보를 만들지 마세요.
3. [컨텍스트]에 질문과 관련된 조문 내용이 전혀 없다면, "제시된 대한민국 헌법 문서에서 해당 내용을 찾을 수 없습니다."라고 답변해주세요.

[컨텍스트]
{context}
"""

# 사용자 프롬프트
user_prompt = "{question}"

# 전체 프롬프트 작성
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", user_prompt)
])

print(prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n당신은 신뢰할 수 있는 대한민국 헌법 전문 법률 Q&A 도우미입니다.\n주어진 헌법 컨텍스트 정보를 바탕으로 사용자의 질문에 정확하고 공손하게 답변해주세요.\n\n규칙:\n1. 답변 시 관련된 헌법 조항(예: 제O조 제O항)이 [컨텍스트]에 명시되어 있다면 반드시 조항 번호를 언급해주세요.\n2. [컨텍스트]의 내용에 근거해서만 답변해야 하며, 자의적인 해석이나 허위 정보를 만들지 마세요.\n3. [컨텍스트]에 질문과 관련된 조문 내용이 전혀 없다면, "제시된 대한민국 헌법 문서에서 해당 내용을 찾을 수 없습니다."라고 답변해주세요.\n\n[컨텍스트]\n{context}\n'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})]


### LLM 생성

In [12]:
# LangChain -> ChatOpenAI 클래스 호출
llm = ChatOpenAI(model='gpt-5-nano')

### outputparser 생성

In [13]:
output_parser = StrOutputParser()

### chain 생성

In [14]:
# LCEL 방식 -> '|' 사용
rag_chain = prompt | llm | output_parser

## chatbot 실행

In [15]:
# 챗봇 실행을 위한 while 반복문 정의 및 실행

## 안내문 출력
print("안녕하세요! 대한민국헌법 Q&A 챗봇입니다. 헌법 조문이나 기본권 등에 대해 질문해주세요. (종료하려면 'exit' 입력)")
print('-' * 80)

## while 조건문 작성
while True:
    try:
        ## 사용자 입력(질문) 생성
        user_query = input("질문: ")

        ## 사용자 입력 -> 종료 조건 만족 -> 종료
        if user_query.lower() == 'exit':
            print("헌법 Q&A 챗봇을 종료합니다. 이용해주셔서 감사합니다.")
            break

        ## 검색 도구를 사용, 사용자 질문과 가장 유사한 Document 2개 -> 리스트 추출
        retrieved_docs = retriever.invoke(user_query)

        ## 검색 결과 -> 텍스트 추출 -> 병합 

        ### 1. 추출한 텍스트를 담을 빈 리스트 생성
        contents = []  

        ### 2. 검색된 2개의 Document 리스트 -> for문 -> Document 별 텍스트 추출 및 contents에 추가
        for doc in retrieved_docs:
            contents.append(doc.page_content)  
        
        ### 3. 문자열 전용 함수 join() 함수 -> 병합 기준: "\n\n"
        context_text = "\n\n".join(contents)

        print(f"[검색된 참조 조문 근거]:\n{context_text}")
        print('-' * 80)

        ## rag_chain.invoke() 함수 실행 -> user_query, context_text 전달
        response = rag_chain.invoke({
            'context': context_text,
            'question': user_query
        })

        print(f"chatbot 응답:\n{response}")
        print('-' * 80)

    except Exception as e:
        print(f'오류가 발생했습니다: {e}')

안녕하세요! 대한민국헌법 Q&A 챗봇입니다. 헌법 조문이나 기본권 등에 대해 질문해주세요. (종료하려면 'exit' 입력)
--------------------------------------------------------------------------------
[검색된 참조 조문 근거]:
③국회의원은 그 지위를 남용하여 국가ㆍ공공단체 또는 기업체와의 계약이나 그 처분에 의하여 재산상의 권리ㆍ이
익 또는 직위를 취득하거나 타인을 위하여 그 취득을 알선할 수 없다.
 
제47조 ①국회의 정기회는 법률이 정하는 바에 의하여 매년 1회 집회되며, 국회의 임시회는 대통령 또는 국회재적의원
4분의 1 이상의 요구에 의하여 집회된다.
②정기회의 회기는 100일을, 임시회의 회기는 30일을 초과할 수 없다.
③대통령이 임시회의 집회를 요구할 때에는 기간과 집회요구의 이유를 명시하여야 한다.
 
제48조 국회는 의장 1인과 부의장 2인을 선출한다.
 
제49조 국회는 헌법 또는 법률에 특별한 규정이 없는 한 재적의원 과반수의 출석과 출석의원 과반수의 찬성으로 의결
한다. 가부동수인 때에는 부결된 것으로 본다.
 
제50조 ①국회의 회의는 공개한다. 다만, 출석의원 과반수의 찬성이 있거나 의장이 국가의 안전보장을 위하여 필요하다
고 인정할 때에는 공개하지 아니할 수 있다.

법제처                                                            5                                                       국가법령정보센터
대한민국헌법
③국회의원의 선거구와 비례대표제 기타 선거에 관한 사항은 법률로 정한다.
 
제42조 국회의원의 임기는 4년으로 한다.
 
제43조 국회의원은 법률이 정하는 직을 겸할 수 없다.
 
제44조 ①국회의원은 현행범인인 경우를 제외하고는 회기 중 국회의 동의없이 체포 또는 구금되지 아니한다.
②국회의원이 회기 전에 체포 또는 구금된 때에는 현행범인이 아닌 한 국회의 요